# Introduction to numerical algorithms
## Practice class 11: Minimization

Today we write simple gradient methods, try the minimizers of SciPy, fit a curve by minimization, and solve the travelling salesman problem with simulated annealing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize

rng = np.random.default_rng(2026)

### Useful commands

| Command | Meaning |
| --- | --- |
| `res = optimize.minimize(f, x0, method="CG")` | Minimize `f` starting from `x0` |
| `optimize.minimize(..., jac=df)` | Give the gradient too |
| `optimize.minimize(..., method="Nelder-Mead")` | Downhill simplex, no derivative needed |
| `optimize.minimize(..., callback=path.append)` | Save the position after each iteration |
| `res.x`, `res.fun`, `res.nit`, `res.nfev` | Position, value, iterations, function calls |
| `optimize.minimize(f, x0, args=(x, y))` | Pass extra arguments to `f(p, x, y)` |
| `gx, gy = np.meshgrid(xs, ys)`, `plt.contourf(gx, gy, F)` | Contour plot of a 2D function |
| `np.linalg.norm(v)` | Length of a vector |

### Task 1: Gradient method in 1D

#### 1a. Fixed step

To find a minimum of $f(x)$ we step against the derivative:

$$x_{n+1}=x_n-\gamma f'(x_n).$$

If $\gamma$ is small enough, $f$ gets smaller at every step. Stop when the step $|x_{n+1}-x_n|$ is smaller than `precision`, or after `max_iters` steps.

Write `simple_gradient`. It gets the derivative `df` and returns the final $x$ and the number of steps.

In [ ]:
def simple_gradient(df, x0, gamma=0.01, precision=1e-6, max_iters=10000):
    x = x0
    iters = 0
    # Work Here!

    return x, iters

Try it on $f(x)=(x+5)^2$ from $x_0=3$. The minimum is at $x=-5$.

Then try $\gamma = 0.01,\ 0.1,\ 0.5,\ 0.9,\ 1.0,\ 1.1$ with `max_iters=1000`.
* Which $\gamma$ gets there in one step? Why? ($f''(x)=2$.)
* What happens if $\gamma\geq1$? Use $x_{n+1}+5=(1-2\gamma)(x_n+5)$.

In [ ]:
df_quad = lambda x: 2 * (x + 5)

# Work Here!

#### 1b. Changing step

Near a minimum $f$ looks like a parabola, $f(x)\approx\beta(x-a)^2$. For a parabola the best step is $\gamma=1/(2\beta)$, which jumps right to $a$. We can estimate it from the last two points:

$$\gamma_n=\left|\frac{x_n-x_{n-1}}{f'(x_n)-f'(x_{n-1})}\right| .$$

Write `gradient_adaptive`. Make the first step with the given `gamma`, then update $\gamma$ after every step. If $f'(x_n)=f'(x_{n-1})$, stop, otherwise you divide by zero.

Compare the number of steps of the two methods for
* $f(x)=(x+5)^2$ from $x_0=3$,
* $f(x)=x^3-x$ from $x_0=3$ (minimum at $1/\sqrt3$).

In [ ]:
def gradient_adaptive(df, x0, gamma=0.01, precision=1e-6, max_iters=10000):
    x = x0
    iters = 0
    # Work Here!

    return x, iters

In [ ]:
df_cubic = lambda x: 3 * x**2 - 1

# Work Here! Print position and number of steps for both methods and both functions.

### Task 2: Minimization in 2D

In more dimensions we step against the gradient:

$$\mathbf x_{n+1}=\mathbf x_n-\gamma_n\nabla F(\mathbf x_n),$$

and the step size is

$$\gamma_n=\frac{\left|(\mathbf x_n-\mathbf x_{n-1})^T\left[\nabla F(\mathbf x_n)-\nabla F(\mathbf x_{n-1})\right]\right|}{\left\|\nabla F(\mathbf x_n)-\nabla F(\mathbf x_{n-1})\right\|^2}.$$

#### 2a. Your own method

Test function:

$$F(x,y)=(x-1)^2+(x+y)^2,$$

with the minimum at $(1,-1)$.

1. Calculate the gradient by hand and write `dF2`.
2. Write `gradient_nd`. It is the same as `gradient_adaptive`, but with vectors (use `np.dot` and `np.linalg.norm`). Save every position in `path` (use `.copy()`).
3. Start from $(3,3)$ and draw the path with `plot_paths`.

In [ ]:
def F2(p):
    x, y = p[0], p[1]
    return (x - 1)**2 + (x + y)**2


def dF2(p):
    x, y = p[0], p[1]
    # Work Here! Return the gradient as np.array([dF/dx, dF/dy]).


def gradient_nd(dF, x0, gamma=0.01, precision=1e-8, max_iters=10000):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    # Work Here!

    return x, np.array(path)


def plot_paths(F, paths, xlim, ylim, log=False, ng=200):
    gx, gy = np.meshgrid(np.linspace(*xlim, ng), np.linspace(*ylim, ng))
    values = F(np.array([gx, gy]))
    if log:
        values = np.log10(values + 1e-3)
    plt.figure(figsize=(6, 5))
    plt.contourf(gx, gy, values, levels=30, cmap="gray")
    plt.colorbar(label="log10 F" if log else "F")
    for label, path in paths.items():
        path = np.array(path)
        plt.plot(path[:, 0], path[:, 1], "o-", ms=3, label=f"{label} ({len(path) - 1} steps)")
    plt.xlim(xlim); plt.ylim(ylim)
    plt.xlabel("x"); plt.ylabel("y"); plt.legend()
    plt.show()

In [ ]:
x_min, path_own = gradient_nd(dF2, [3, 3])
print("Minimum:", x_min, " steps:", len(path_own) - 1)
plot_paths(F2, {"own gradient": path_own}, xlim=(-1, 3.5), ylim=(-2, 3.5))

#### 2b. SciPy

Two methods of `optimize.minimize`:

* **CG** (conjugate gradient) uses the gradient. Without `jac` it calculates the gradient numerically, which needs extra function calls.
* **Nelder-Mead** (downhill simplex, see the lecture) uses only function values. It moves a triangle towards the minimum.

Minimize $F$ from $(3,3)$ with CG, CG with `jac=dF2`, and Nelder-Mead. Save the paths with `callback` (put the starting point in the list first). Print `res.x`, `res.nit` and `res.nfev`, and draw all paths together with yours.

Which one needs the fewest function calls? Why is the number of iterations not enough to compare them?

In [ ]:
# Work Here!

### Task 3: Fitting is minimization

In class 9 we used `curve_fit`. It minimizes the squared error

$$E(\mathbf p)=\sum_i\bigl(y_i-\hat y(x_i;\mathbf p)\bigr)^2 .$$

Now do it with `optimize.minimize`. The data are

$$y=A\sin\bigl(\omega(x+\varphi)\bigr)+\text{noise},\qquad A=1.3,\ \omega=2,\ \varphi=0.5.$$

1. Write `loss(p, x, y)` with `p = [A, omega, phi]`, without a loop.
2. Minimize it with `optimize.minimize(loss, x0, args=(x_sin, y_sin), method="CG")` from `x0 = [1, 2, 0]`. Plot the data and the fit.
3. Also start from `[1, 1, 0]` and `[1, 2, 2]`. Print the parameters and $E$, and plot the fits.
   * One of them gets stuck in a local minimum. The fit is bad and $E$ is large.
   * The other fits well, but with other parameters. Why do $(A,\omega,\varphi)$, $(-A,\omega,\varphi+\pi/\omega)$ and $(A,\omega,\varphi+2\pi/\omega)$ give the same curve?

In [ ]:
x_sin = np.linspace(0, 10, 200)
y_sin = 1.3 * np.sin(2 * (x_sin + 0.5)) + rng.normal(0, 0.2, x_sin.size)


def loss(p, x, y):
    # Work Here!


# Work Here!

### Task 4: Local minima

The function

$$f(x)=x^6-4x^4+5x^2,\qquad f'(x)=6x^5-16x^3+10x$$

has its global minimum at $x=0$ and two local minima at $x=\pm\sqrt{5/3}\approx\pm1.29$.

#### 4a. Where do we end up?

Plot $f$ on $[-2,2]$. Run `gradient_adaptive` and `optimize.minimize(..., method="CG")` from $x_0 = 3,\ 1.5,\ 0.6,\ 0.3$. Which minimum do they find? Explain it with the plot.

In [ ]:
f6 = lambda x: x**6 - 4 * x**4 + 5 * x**2
df6 = lambda x: 6 * x**5 - 16 * x**3 + 10 * x

# Work Here!

#### 4b. Simulated annealing

The gradient methods only go down, so they can get stuck. Simulated annealing sometimes accepts a step upwards. At high temperature it moves almost freely, and as we cool it, it settles in a deep minimum.

1. Move: $x_\mathrm{new}=x+\delta$, where $\delta$ is uniform in $[-d,d)$.
2. Let $\Delta f=f(x_\mathrm{new})-f(x)$.
   * If $\Delta f\leq0$, accept the move.
   * If $\Delta f>0$, accept it with probability $e^{-\Delta f/T}$: accept if `rng.random() < np.exp(-df/T)`. Otherwise stay at $x$.
3. Cooling: step $k$ uses `temperatures[k]`. Make the temperatures with `np.geomspace(T0, T_end, n_steps)`.

Write `annealing(f, x0, temperatures, d, rng)`. It returns the positions after every step.

Run it from $x_0=3$ with $T_0=5$, $T_\mathrm{end}=10^{-3}$, $10^4$ steps and $d=0.1$, and plot $x$ against the step number. Run it 20 times. How many times does it end near $0$ ($|x|<0.3$)?

What happens with only 200 steps?

In [ ]:
def annealing(f, x0, temperatures, d, rng):
    x = x0
    fx = f(x)
    path = [x]
    for T in temperatures:
        # Work Here!

        path.append(x)
    return np.array(path)

In [ ]:
# Work Here!

### Task 5: Travelling salesman

We put $N$ random cities in the unit square. The salesman visits each city once and goes back to the start, in straight lines. We want a short route. There are $(N-1)!/2$ possible routes, so we cannot check all of them.

The route is an array `order` with the city indices in the order of visit. `tour_length` and `plot_tour` are given.

**The move (2-opt):** the best route never crosses itself. Pick two positions $i<j$, and reverse the part `order[i+1 : j+1]`. This replaces the edges `order[i] → order[i+1]` and `order[j] → order[j+1]` with `order[i] → order[j]` and `order[i+1] → order[j+1]`. In the picture the red route crosses itself, and the green one is the result of this move.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot([0, 1, 1.5, 1, 0, 0], [0, 1, 0.5, 0, 1, 0], "r*-", label="crossing")
plt.plot([-0.01, 1.01, 1.53, 1.01, -0.01, -0.01], [-0.01, -0.01, 0.5, 1.01, 1.01, -0.01], "g*-", label="after 2-opt")
plt.legend(); plt.axis("off")
plt.show()

1. Write `two_opt(order, i, j)`. It returns a new array where `order[i+1 : j+1]` is reversed (`[::-1]`). Assume `i < j`.
2. Write `tsp_search(cities, order, temperatures, rng)`. At each step pick two different random positions, sort them, try `two_opt`, and accept or reject it as in 4b. Return the final order and the list of route lengths.
3. With `temperatures = np.zeros(n_steps)` only shorter (or equal) routes are accepted. This is the greedy method. Run greedy and annealing for $N=40$ and $40\,000$ steps (annealing: $T_0=0.5$, $T_\mathrm{end}=10^{-3}$), from the same random route. Plot the length against the step number, and both final routes. Which one is shorter?

In [ ]:
N_cities = 40
cities = rng.random((N_cities, 2))


def tour_length(order, cities):
    pts = cities[order]
    return np.sum(np.linalg.norm(pts - np.roll(pts, -1, axis=0), axis=1))


def plot_tour(order, cities, ax, title=""):
    pts = cities[np.append(order, order[0])]
    ax.plot(pts[:, 0], pts[:, 1], "o-", ms=4)
    ax.set(title=f"{title} L = {tour_length(order, cities):.3f}", aspect="equal")


def two_opt(order, i, j):
    # Work Here!


def tsp_search(cities, order, temperatures, rng):
    order = order.copy()
    L = tour_length(order, cities)
    lengths = []
    n = len(order)
    for T in temperatures:
        # Work Here!

        lengths.append(L)
    return order, lengths

In [ ]:
# Work Here!